# Defining the subevent matching algorithm

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import uproot
import sys
import seaborn as sns
from tqdm import tqdm
import networkx as nx

import atlasify as atl
from particle import Particle
atl.ATLAS = "ColliderML"

sys.path.append("/global/cfs/cdirs/m4958/usr/danieltm/ColliderML/software/OtherLibraries/pyedm4hep")
from pyedm4hep import EDM4hepEvent
sys.path.append("/global/cfs/cdirs/m4958/usr/danieltm/ColliderML/software/colliderml_dev/notebooks")
from loading_utils import load_root_file, load_hepmc_event

## Roadmap

1. Load an ACTS particles file
2. Examine primary vertices...


## Loading

In [13]:
import awkward as ak

In [22]:
input_file = "/pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_mini_pilot/ttbar/v6/runs/0/particles.root"
particles = load_root_file(input_file, event_id=0, ignore_variable_columns=False)
# Remove the multiindex
particles = particles.reset_index(drop=True)
particles

,event_id,particle_id,particle_type,process,vx,vy,vz,vt,px,py,...,vertex_primary,vertex_secondary,particle,generation,sub_particle,e_loss,total_x0,total_l0,number_of_hits,outcome
0,0,4503599627370496,413,0,-0.002071,-0.003950,-40.149307,-373.873352,8.362612,-5.792818,...,1,0,0,0,0,22.365232,0.0,0.0,0,0
1,0,4503599627370497,313,0,-0.002071,-0.003950,-40.149307,-373.873352,5.551582,-3.833765,...,1,0,0,0,1,0.000000,0.0,0.0,0,0
2,0,4503599627370498,-311,0,-0.002071,-0.003950,-40.149307,-373.873352,1.093668,0.147071,...,1,0,0,0,2,0.000000,0.0,0.0,0,0
3,0,4503599627370499,-211,0,-0.002071,-0.003950,-40.149307,-373.873352,1.041120,-2.201525,...,1,0,0,0,3,4.084134,0.0,0.0,11,0
4,0,4503599627370500,213,0,-0.002071,-0.003950,-40.149307,-373.873352,1.218392,-1.988743,...,1,0,0,0,4,0.000000,0.0,0.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82835,0,905392849892344261,11,0,-47.749573,-320.650299,1842.503784,29301.808594,-0.000090,-0.000423,...,201,154,0,3,453,0.000476,0.0,0.0,2,0
82836,0,905392849892344262,11,0,-46.229919,-426.554230,2545.861328,33254.269531,0.000180,-0.001465,...,201,154,0,3,454,0.001290,0.0,0.0,2,0
82837,0,905396148427161901,-211,0,5.574176,50.942642,619.227478,-943.589478,0.384727,0.449388,...,201,157,0,2,301,2.268601,0.0,0.0,10,0
82838,0,905396148427161902,2212,0,5.574176,50.942642,619.227478,-943.589478,-0.346143,-0.278581,...,201,157,0,2,302,0.116091,0.0,0.0,6,0


In [23]:
particles

,event_id,particle_id,particle_type,process,vx,vy,vz,vt,px,py,...,vertex_primary,vertex_secondary,particle,generation,sub_particle,e_loss,total_x0,total_l0,number_of_hits,outcome
0,0,4503599627370496,413,0,-0.002071,-0.003950,-40.149307,-373.873352,8.362612,-5.792818,...,1,0,0,0,0,22.365232,0.0,0.0,0,0
1,0,4503599627370497,313,0,-0.002071,-0.003950,-40.149307,-373.873352,5.551582,-3.833765,...,1,0,0,0,1,0.000000,0.0,0.0,0,0
2,0,4503599627370498,-311,0,-0.002071,-0.003950,-40.149307,-373.873352,1.093668,0.147071,...,1,0,0,0,2,0.000000,0.0,0.0,0,0
3,0,4503599627370499,-211,0,-0.002071,-0.003950,-40.149307,-373.873352,1.041120,-2.201525,...,1,0,0,0,3,4.084134,0.0,0.0,11,0
4,0,4503599627370500,213,0,-0.002071,-0.003950,-40.149307,-373.873352,1.218392,-1.988743,...,1,0,0,0,4,0.000000,0.0,0.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82835,0,905392849892344261,11,0,-47.749573,-320.650299,1842.503784,29301.808594,-0.000090,-0.000423,...,201,154,0,3,453,0.000476,0.0,0.0,2,0
82836,0,905392849892344262,11,0,-46.229919,-426.554230,2545.861328,33254.269531,0.000180,-0.001465,...,201,154,0,3,454,0.001290,0.0,0.0,2,0
82837,0,905396148427161901,-211,0,5.574176,50.942642,619.227478,-943.589478,0.384727,0.449388,...,201,157,0,2,301,2.268601,0.0,0.0,10,0
82838,0,905396148427161902,2212,0,5.574176,50.942642,619.227478,-943.589478,-0.346143,-0.278581,...,201,157,0,2,302,0.116091,0.0,0.0,6,0


In [24]:
particles.columns

Index(['event_id', 'particle_id', 'particle_type', 'process', 'vx', 'vy', 'vz',
       'vt', 'px', 'py', 'pz', 'm', 'q', 'eta', 'phi', 'pt', 'p',
       'vertex_primary', 'vertex_secondary', 'particle', 'generation',
       'sub_particle', 'e_loss', 'total_x0', 'total_l0', 'number_of_hits',
       'outcome'],
      dtype='object')

In [28]:
particles

,event_id,particle_id,particle_type,process,vx,vy,vz,vt,px,py,...,vertex_primary,vertex_secondary,particle,generation,sub_particle,e_loss,total_x0,total_l0,number_of_hits,outcome
0,0,4503599627370496,413,0,-0.002071,-0.003950,-40.149307,-373.873352,8.362612,-5.792818,...,1,0,0,0,0,22.365232,0.0,0.0,0,0
1,0,4503599627370497,313,0,-0.002071,-0.003950,-40.149307,-373.873352,5.551582,-3.833765,...,1,0,0,0,1,0.000000,0.0,0.0,0,0
2,0,4503599627370498,-311,0,-0.002071,-0.003950,-40.149307,-373.873352,1.093668,0.147071,...,1,0,0,0,2,0.000000,0.0,0.0,0,0
3,0,4503599627370499,-211,0,-0.002071,-0.003950,-40.149307,-373.873352,1.041120,-2.201525,...,1,0,0,0,3,4.084134,0.0,0.0,11,0
4,0,4503599627370500,213,0,-0.002071,-0.003950,-40.149307,-373.873352,1.218392,-1.988743,...,1,0,0,0,4,0.000000,0.0,0.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82835,0,905392849892344261,11,0,-47.749573,-320.650299,1842.503784,29301.808594,-0.000090,-0.000423,...,201,154,0,3,453,0.000476,0.0,0.0,2,0
82836,0,905392849892344262,11,0,-46.229919,-426.554230,2545.861328,33254.269531,0.000180,-0.001465,...,201,154,0,3,454,0.001290,0.0,0.0,2,0
82837,0,905396148427161901,-211,0,5.574176,50.942642,619.227478,-943.589478,0.384727,0.449388,...,201,157,0,2,301,2.268601,0.0,0.0,10,0
82838,0,905396148427161902,2212,0,5.574176,50.942642,619.227478,-943.589478,-0.346143,-0.278581,...,201,157,0,2,302,0.116091,0.0,0.0,6,0
